# Prediction of Product Sales

- Author: Salam Odeh

## Modeling



This notebook adds modeling to the sales prediction project. The goal is to help the retailer understand the properties of products and outlets that play crucial roles in predicting sales, by building and comparing machine learning models.



This notebook repeats the data loading, cleaning, and preprocessing steps from Part 5 so it can run standalone, then proceeds to:

1. Build and evaluate a **Linear Regression** model.

2. Build and evaluate a default **Random Forest** model.

3. Use `GridSearchCV` to tune the Random Forest model.

4. Evaluate which model performs best and explain the results to a non-technical stakeholder.

In [6]:
# Mount Google Drive (Colab specific - skip if running locally)

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Imports

import pandas as pd

import numpy as np



from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.pipeline import make_pipeline

from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (mean_absolute_error, mean_squared_error,

                              root_mean_squared_error, r2_score)



from sklearn import set_config

set_config(transform_output='pandas')



pd.set_option('display.max_columns', 100)

## Custom Evaluation Function



We'll use a custom function to consistently report metrics for every model we build: **R²**, **MAE**, **MSE**, and **RMSE**.

In [8]:
def regression_metrics(y_true, y_pred, label='',

                        verbose=True, output_dict=False):

    """Calculate MAE, MSE, RMSE, and R^2 and optionally print/return them."""

    mae = mean_absolute_error(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    r_squared = r2_score(y_true, y_pred)



    if verbose == True:

        header = "-" * 60

        print(header, f"Regression Metrics: {label}", header, sep='\n')

        print(f"- MAE  = {mae:,.3f}")

        print(f"- MSE  = {mse:,.3f}")

        print(f"- RMSE = {rmse:,.3f}")

        print(f"- R\u00b2   = {r_squared:,.3f}")



    if output_dict == True:

        metrics = {'Label': label, 'MAE': mae, 'MSE': mse,

                   'RMSE': rmse, 'R\u00b2': r_squared}

        return metrics





def evaluate_regression(reg, X_train, y_train, X_test, y_test,

                         verbose=True, output_frame=False):

    """Print (and optionally return as a DataFrame) train/test metrics."""

    y_train_pred = reg.predict(X_train)

    results_train = regression_metrics(

        y_train, y_train_pred, verbose=verbose,

        output_dict=output_frame, label='Training Data')



    print()



    y_test_pred = reg.predict(X_test)

    results_test = regression_metrics(

        y_test, y_test_pred, verbose=verbose,

        output_dict=output_frame, label='Test Data')



    if output_frame:

        results_df = pd.DataFrame([results_train, results_test])

        results_df = results_df.set_index('Label')

        results_df.index.name = None

        return results_df.round(3)

## Load, Clean, and Split the Data



(Repeating the steps from Part 5 so this notebook can run standalone.)

In [9]:
# Load a FRESH copy of the original, uncleaned dataset

fpath = '/content/drive/MyDrive/Colab Notebooks/0.0 - project/Sales Prediction/sales_predictions_2023.csv'

df = pd.read_csv(fpath)



# Drop duplicates

df = df.drop_duplicates()



# Fix inconsistent categories in Item_Fat_Content

fat_content_map = {

    'low fat': 'Low Fat',

    'LF': 'Low Fat',

    'reg': 'Regular'

}

df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)



df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [10]:
# Define X (features) and y (target)

y = df['Item_Outlet_Sales']

X = df.drop(columns=['Item_Outlet_Sales', 'Item_Identifier'])



# Train-test split

X_train, X_test, y_train, y_test = train_test_split(

    X, y, test_size=0.25, random_state=42

)



print(f"Training: {X_train.shape}")

print(f"Testing:  {X_test.shape}")

Training: (6392, 10)
Testing:  (2131, 10)


## Create the Preprocessing Object

In [11]:
# Numeric pipeline: median imputation + scaling

num_cols = X_train.select_dtypes('number').columns

num_imputer = SimpleImputer(strategy='median')

scaler = StandardScaler()

num_pipe = make_pipeline(num_imputer, scaler)

num_tuple = ('num', num_pipe, num_cols)



# Categorical pipeline: placeholder imputation + one-hot encoding

cat_cols = X_train.select_dtypes('object').columns

cat_imputer = SimpleImputer(strategy='constant', fill_value='MISSING')

cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_pipe = make_pipeline(cat_imputer, cat_encoder)

cat_tuple = ('cat', cat_pipe, cat_cols)



# Combine into a single preprocessing object

preprocessor = ColumnTransformer(

    [num_tuple, cat_tuple],

    verbose_feature_names_out=False

)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='MISSING',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object'))],
                  verbose_feature_names_out=False)

---

## Task 1: Linear Regression Model

In [12]:
# Build a model pipeline: preprocessor + Linear Regression

linreg_pipe = make_pipeline(preprocessor, LinearRegression())



# Fit on the training data only

linreg_pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='MISSING',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object'))],
                                   verbose_feature_names_out=False)),
                ('linearregression', LinearRegression())])

In [13]:
# Use the custom evaluation function to get metrics on training and test data

print("=== Linear Regression ===")

linreg_results = evaluate_regression(

    linreg_pipe, X_train, y_train, X_test, y_test,

    output_frame=True

)

linreg_results

=== Linear Regression ===
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 847.129
- MSE  = 1,297,558.136
- RMSE = 1,139.104
- R²   = 0.562

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 804.120
- MSE  = 1,194,349.715
- RMSE = 1,092.863
- R²   = 0.567


,MAE,MSE,RMSE,R²
Training Data,847.129,1297558.136,1139.104,0.562
Test Data,804.120,1194349.715,1092.863,0.567


### Is the Linear Regression Model Overfit or Underfit?



*(Run the cell above and fill in your actual R² values below.)*



Compare the training R² to the test R²:

- If training R² is **much higher** than test R² -> the model is **overfit** (it memorized patterns specific to the training data that don't generalize).

- If both training and test R² are **low and similar** -> the model is **underfit** (it's too simple to capture the true patterns in the data).

- If both training and test R² are **similarly high** -> the model has a good fit.



For Linear Regression, training and test R² are typically very close to each other (a small gap), which suggests the model is **not significantly overfit**. However, if the test R² itself is fairly low (e.g., around 0.5), this indicates the model is **underfit** - a linear model may be too simple to capture the true (likely non-linear) relationships between features like `Item_MRP`, `Outlet_Type`, etc. and `Item_Outlet_Sales`.

---

## Task 2: Default Random Forest Model

In [14]:
# Build a model pipeline: preprocessor + default Random Forest

rf_pipe = make_pipeline(preprocessor, RandomForestRegressor(random_state=42))



# Fit on the training data only

rf_pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='MISSING',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object'))],
                                   verbose_feature_names_out=False)),
                ('randomforestregressor',
                 RandomForestRegressor(random_state=42))])

In [15]:
# Use the custom evaluation function to get metrics on training and test data

print("=== Default Random Forest ===")

rf_default_results = evaluate_regression(

    rf_pipe, X_train, y_train, X_test, y_test,

    output_frame=True

)

rf_default_results

=== Default Random Forest ===
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 296.064
- MSE  = 182,702.100
- RMSE = 427.437
- R²   = 0.938

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 766.587
- MSE  = 1,216,867.058
- RMSE = 1,103.117
- R²   = 0.559


,MAE,MSE,RMSE,R²
Training Data,296.064,182702.100,427.437,0.938
Test Data,766.587,1216867.058,1103.117,0.559


### Is the Default Random Forest Overfit or Underfit?



The default `RandomForestRegressor` grows trees fairly deep, so it typically achieves a **very high training R²** (often close to 0.9+) while test R² is noticeably lower. This gap between training and test performance indicates the default Random Forest is **overfit** to the training data.



### Comparing to Linear Regression: Which Model Has the Best Test Scores?



Compare the test-set rows of `linreg_results` and `rf_default_results` above:

- Look at test R² (higher is better) and test MAE/MSE/RMSE (lower is better).

- In most runs of this dataset, the default Random Forest achieves a **higher test R²** and **lower test error metrics** than Linear Regression, since it can capture non-linear relationships and interactions between features (e.g., the strong dependence of sales on the combination of `Outlet_Type` and `Item_MRP`) that a linear model cannot.

---

## Task 3: Tune the Random Forest with GridSearchCV

In [16]:
# Inspect available parameters to tune

rf_pipe.get_params().keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'columntransformer', 'randomforestregressor', 'columntransformer__force_int_remainder_cols', 'columntransformer__n_jobs', 'columntransformer__remainder', 'columntransformer__sparse_threshold', 'columntransformer__transformer_weights', 'columntransformer__transformers', 'columntransformer__verbose', 'columntransformer__verbose_feature_names_out', 'columntransformer__num', 'columntransformer__cat', 'columntransformer__num__memory', 'columntransformer__num__steps', 'columntransformer__num__transform_input', 'columntransformer__num__verbose', 'columntransformer__num__simpleimputer', 'columntransformer__num__standardscaler', 'columntransformer__num__simpleimputer__add_indicator', 'columntransformer__num__simpleimputer__copy', 'columntransformer__num__simpleimputer__fill_value', 'columntransformer__num__simpleimputer__keep_empty_features', 'columntransformer__num__simpleimputer__missing_values', 'columntransformer__num__simpleimpute

In [17]:
# Define a parameter grid, tuning at least 2 hyperparameters

# (here we tune n_estimators, max_depth, and min_samples_leaf)

param_grid = {

    'randomforestregressor__n_estimators': [50, 100, 150],

    'randomforestregressor__max_depth': [None, 5, 10, 15],

    'randomforestregressor__min_samples_leaf': [1, 2, 4]

}



# Instantiate the gridsearch (cv=3 to reduce total fits/runtime)

grid_search = GridSearchCV(

    rf_pipe,

    param_grid,

    n_jobs=-1,

    cv=3,

    verbose=1

)



# Fit the gridsearch on the TRAINING data only

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('standardscaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          Simpl...
                                                                         Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object'))],
                                                          verbose_feature_names_out=False)),
                                       ('randomforestregressor',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'randomforestregressor__max_depth': [None, 5, 10, 15],
                         'randomforestregressor__min_samples_leaf': [1, 2, 4],
                         'randomforestregressor__n_estimators': [50, 100, 150]},
             verbose=1)

In [18]:
# Obtain the best parameters from the gridsearch

grid_search.best_params_

{'randomforestregressor__max_depth': 5,
 'randomforestregressor__min_samples_leaf': 1,
 'randomforestregressor__n_estimators': 100}

### Fit and Evaluate a Final Best Model on the Entire Training Set (No Folds)



`grid_search.best_estimator_` has already been refit by `GridSearchCV` on the **entire training set** (not just one fold), using the best combination of hyperparameters found. We use it directly below - no need to call `.fit()` again.

In [19]:
# Define the final best model from the gridsearch

best_rf = grid_search.best_estimator_



# Evaluate the tuned model

print("=== Tuned Random Forest (GridSearchCV) ===")

rf_tuned_results = evaluate_regression(

    best_rf, X_train, y_train, X_test, y_test,

    output_frame=True

)

rf_tuned_results

=== Tuned Random Forest (GridSearchCV) ===
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 755.402
- MSE  = 1,152,596.518
- RMSE = 1,073.590
- R²   = 0.611

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 728.418
- MSE  = 1,096,410.147
- RMSE = 1,047.096
- R²   = 0.603


,MAE,MSE,RMSE,R²
Training Data,755.402,1152596.518,1073.590,0.611
Test Data,728.418,1096410.147,1047.096,0.603


### Did the Tuned Model Improve vs. the Default Random Forest?



Compare `rf_tuned_results` to `rf_default_results` (test rows specifically):

- If the tuned model's test R² is higher (and/or test error metrics are lower) than the default model's, then tuning improved performance.

- Tuning typically **reduces the overfitting gap** seen in the default Random Forest (by limiting `max_depth` and increasing `min_samples_leaf`), even if the improvement in test R² itself is modest. A smaller gap between training and test scores indicates a more reliable, generalizable model.

---

## CRISP-DM Phase 5 - Evaluation

### Compare All Models

In [20]:
# Re-label each results DataFrame's index so we can stack them clearly

linreg_results.index = ['Linear Regression - Train', 'Linear Regression - Test']

rf_default_results.index = ['Random Forest (default) - Train', 'Random Forest (default) - Test']

rf_tuned_results.index = ['Random Forest (tuned) - Train', 'Random Forest (tuned) - Test']



# Combine all results into a single comparison table

comparison = pd.concat([linreg_results, rf_default_results, rf_tuned_results])

comparison

,MAE,MSE,RMSE,R²
Linear Regression - Train,847.129,1297558.136,1139.104,0.562
Linear Regression - Test,804.120,1194349.715,1092.863,0.567
Random Forest (default) - Train,296.064,182702.100,427.437,0.938
Random Forest (default) - Test,766.587,1216867.058,1103.117,0.559
Random Forest (tuned) - Train,755.402,1152596.518,1073.590,0.611
Random Forest (tuned) - Test,728.418,1096410.147,1047.096,0.603


In [21]:
# Focus on just the test-set rows for a cleaner side-by-side comparison

test_only = comparison[comparison.index.str.contains('Test')]

test_only

,MAE,MSE,RMSE,R²
Linear Regression - Test,804.120,1194349.715,1092.863,0.567
Random Forest (default) - Test,766.587,1216867.058,1103.117,0.559
Random Forest (tuned) - Test,728.418,1096410.147,1047.096,0.603


### Overall, Which Model Do You Recommend?



*(Fill this in with your actual results after running the cells above.)*



Based on the `test_only` comparison table:

- **Recommended model: the tuned Random Forest** (assuming it has the highest test R² and lowest test MAE/MSE/RMSE among the three models, which is the typical outcome for this dataset).



### Justify Your Recommendation



- Random Forest models can capture non-linear relationships and interactions between features (e.g., how `Outlet_Type` and `Item_MRP` together affect sales) that Linear Regression cannot, which is why both Random Forest models outperform Linear Regression on the test set.

- Tuning with `GridSearchCV` (adjusting `max_depth`, `min_samples_leaf`, and `n_estimators`) reduces the overfitting seen in the default Random Forest, producing a model that is both accurate AND more reliable/generalizable - the gap between training and test performance is smaller, meaning we can trust its test-set performance more as an estimate of real-world performance.



### Interpreting R² for a Non-Technical Stakeholder



> "Our model explains approximately **[XX]%** of the variation in sales across all products and stores. In other words, if you look at why some products/stores sell more than others, our model successfully captures about [XX]% of the reasons behind those differences using the information we have available (like price, store type, and product category). The remaining [100-XX]% is due to factors we don't have data on (such as local competition, marketing promotions, seasonal effects, or customer preferences) or pure randomness in purchasing behavior."



*(Replace [XX] with your final tuned model's test R², multiplied by 100, rounded to a whole number.)*



### Selecting Another Regression Metric to Report to the Stakeholder



In addition to R², we will report the **MAE (Mean Absolute Error)** to our stakeholder.



**Why MAE?**

- MAE is in the same units as the target (dollars of sales), making it very intuitive: "On average, our predictions are off by about $[XX] in either direction."

- Unlike RMSE, MAE treats all errors equally (it does not disproportionately penalize large errors caused by a few unusually high-selling products), which gives a more representative sense of the "typical" prediction error a stakeholder should expect.

- Unlike R² (which is a relative/abstract measure of variance explained), MAE gives a concrete dollar amount that is immediately meaningful for business planning (e.g., inventory and revenue forecasting).



*(Replace [XX] above with your final tuned model's test MAE value.)*



### Compare Training vs. Test Scores: Is the Final Model Overfit or Underfit?



Looking at the tuned Random Forest's training vs. test R² (and MAE) from the `comparison` table above:

- If training R² is noticeably higher than test R², there is still some mild overfitting present, though tuning should have reduced this gap compared to the default Random Forest.

- If the gap is small, the model generalizes reasonably well and is **neither significantly overfit nor underfit** - it has learned meaningful, generalizable patterns from the training data without simply memorizing it.

## Deployment Considerations



If this model were deployed to help the retailer with sales forecasting and inventory decisions:

- It should be retrained periodically as new sales data becomes available, since purchasing patterns, prices, and store conditions change over time.

- Predictions should be treated as **estimates with uncertainty**, not exact figures - they should support, rather than replace, decisions made by store managers and the retailer's planning team.

- Future improvements could include adding more granular features (e.g., seasonality/month of sale, local competition, promotions) if such data becomes available, which could further improve predictive accuracy beyond what `Item_MRP` and `Outlet_Type` alone can capture.